In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import classification_report
print("OK")

OK


In [2]:
import sys
sys.path.append("..")

from src.features import extract_symptoms
import pandas as pd

# load data
df_raw = pd.read_excel("../data/raw/covid.xlsx")

# run function
df_symptoms = extract_symptoms(df_raw)

# check output
df_symptoms.head()

,Fever,Myalgia,Chills,Fatigue,Sweating,Anorexia,Sore_Throat,Sneezing,Cough,Dyspnea,...,Ageusia,Chest_Pain,Palpitations,Nausea,Vomiting,Abdominal_Pain,Diarrhea,Depression,Sleep_Disorder,Headache
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0
4,0,0,1,0,0,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [3]:
import sys
sys.path.append("..")

#import importlib
import src.features as features

#importlib.reload(features)

df_pmh = features.extract_pmh(df_raw)

print(df_pmh.shape)
df_pmh.head()

(16780, 28)


,Cancer,Organ_Transplant,Chronic_Kidney_Disease,Dialysis,Cancer_Type,Solid_Organ_Transplant,Asthma,Chronic_Respiratory_Disease,Hypertension,Myocardial_Infarction,...,Seizure,Diabetes_Type1,Diabetes_Type2,Hypothyroidism,Hyperthyroidism,Insulin_Use,Rheumatoid_Arthritis,Lupus,Behcet,Sjogren
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
#import importlib
import src.features as features

#importlib.reload(features)

df_vax = features.extract_vaccination(df_raw)

print(df_vax.shape)
df_vax.head()

(16780, 2)


,Vaccinated,Vaccine_Doses
0,0,0.0
1,0,0.0
2,0,0.0
3,0,0.0
4,1,2.0


In [5]:
print("Vaccinated %:", round(df_vax['Vaccinated'].mean() * 100, 2))
df_vax['Vaccine_Doses'].value_counts().sort_index()

Vaccinated %: 3.43


Vaccine_Doses
0.0    16205
1.0      107
2.0      354
3.0      106
4.0        8
Name: count, dtype: int64

In [6]:
import importlib
import src.features as features

importlib.reload(features)

df_final = features.build_features(df_raw)

print(df_final.shape)
df_final.head()

(16780, 55)


,Fever,Myalgia,Chills,Fatigue,Sweating,Anorexia,Sore_Throat,Sneezing,Cough,Dyspnea,...,Diabetes_Type2,Hypothyroidism,Hyperthyroidism,Insulin_Use,Rheumatoid_Arthritis,Lupus,Behcet,Sjogren,Vaccinated,Vaccine_Doses
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
2,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0.0
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0.0
4,0,0,1,0,0,1,1,0,1,1,...,0,0,0,0,0,0,0,0,1,2.0


In [7]:
# 🎯 Step 1: Create target (CT lung involvement → severity)

df_target = df_raw['Unnamed: 69'].iloc[3:].reset_index(drop=True)

# Convert to binary:
# if there is any text → 1 (lung involvement)
# if empty → 0 (no involvement)

df_target = df_target.notna().astype(int)

# Check distribution
print("Target distribution:")
print(df_target.value_counts())

Target distribution:
Unnamed: 69
0    16588
1      192
Name: count, dtype: int64


In [13]:
#  feature و target

X = df_final.copy()
y = df_target

X = X.loc[y.index]

print(X.shape, y.shape)

(16780, 55) (16780,)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_res.value_counts())

Before SMOTE:
 Unnamed: 69
0    13270
1      154
Name: count, dtype: int64
After SMOTE:
 Unnamed: 69
0    13270
1    13270
Name: count, dtype: int64


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=1000)

model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.69      0.82      3318
           1       0.02      0.42      0.03        38

    accuracy                           0.69      3356
   macro avg       0.50      0.56      0.42      3356
weighted avg       0.98      0.69      0.81      3356

